In [1]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
import torch
from evaluate import load
import numpy as np
import random, re, string
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from datasets import load_dataset, concatenate_datasets
import os

### Augment

In [2]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /mnt/export/arun/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /mnt/export/arun/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
stop_words = set(stopwords.words('english'))

def get_synonyms(word):
    """Return a list of synonyms for a word from WordNet (excluding itself)."""
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            name = lemma.name().replace('_', ' ')
            if name.lower() != word.lower():
                syns.add(name)
    return list(syns)

def synonym_replacement_rate(text, rate=0.01):
    """
    Replace approximately `rate` fraction of non-stopwords in the text with synonyms.

    Args:
        text (str): Input text (sentence or larger).
        rate (float): Fraction of replaceable words to swap out (default 0.01 = 1%).

    Returns:
        str: Augmented text.
    """
    words = word_tokenize(text)
    candidates = [i for i, w in enumerate(words)
                  if w.isalpha() and w.lower() not in stop_words]
    
    n_replace = int(len(candidates) * rate)
    if n_replace < 1:
        return text  
    random.shuffle(candidates)
    replaced = 0
    for idx in candidates:
        syns = get_synonyms(words[idx])
        if syns:
            words[idx] = random.choice(syns)
            replaced += 1
        if replaced >= n_replace:
            break

    return ' '.join(words)

def aug_html_entities(text: str, p=0.1) -> str:
    entities = {"'": "&#39;", '"': "&quot;", "&": "&amp;"}
    for ch, ent in entities.items():
        if random.random() < p:
            text = text.replace(ch, ent)
    return text

def aug_html_entities(text: str, p=0.1) -> str:
    entities = {"'": "&#39;", '"': "&quot;", "&": "&amp;"}
    for ch, ent in entities.items():
        if random.random() < p:
            text = text.replace(ch, ent)
    return text

def aug_word_dup(text: str, p=0.05) -> str:
    words = text.split()
    if words and random.random() < p:
        i = random.randrange(len(words))
        words.insert(i, words[i])
    return " ".join(words)

def aug_case_swap(text: str, p=0.1) -> str:
    return "".join(c.upper() if random.random() < p else c.lower() for c in text)

def aug_punct_space(text: str, p=0.05) -> str:
    out = []
    for c in text:
        if c.isalnum() and random.random() < p:
            out.append(c + random.choice(string.punctuation))
        else:
            out.append(c)
    s = "".join(out)
    return re.sub(r" ", lambda m: " " + (" " if random.random() < p else ""), s)

def aug_truncate(text: str, p=0.1) -> str:
    if random.random() < p and len(text) > 20:
        cut = int(len(text) * random.uniform(0.7, 0.9))
        return text[:cut]
    return text

def aug_char_swap(text: str, p=0.02) -> str:
    chars = list(text)
    for i in range(len(chars) - 1):
        if random.random() < p:
            chars[i], chars[i+1] = chars[i+1], chars[i]
    return "".join(chars)

def augment_text(text: str) -> str:
    aug_funcs = [
        aug_html_entities,
        aug_word_dup,
        aug_case_swap,
        aug_punct_space,
        aug_truncate,
        aug_char_swap,
        synonym_replacement_rate
    ]

    n = random.randint(1, 5)
    chosen = random.sample(aug_funcs, k=n)

    for fn in chosen:
        text = fn(text)
    return text

### Data

In [4]:
def get_augmented_agnews_data(num_samples):
    agnews = load_dataset("ag_news")
    class_names = agnews["train"].features["label"].names
    id2label = {i: name for i, name in enumerate(class_names)}
    label2id = {name: i for i, name in enumerate(class_names)}

    test_data = agnews["test"]
    train_data = agnews["train"]

    labels = sorted(set(train_data["label"]))
    per_label = num_samples // len(labels)
    indices_by_label = {lab: [] for lab in labels}

    for i, lab in enumerate(train_data["label"]):
        indices_by_label[lab].append(i)
    
    sampled_indices = []
    for lab in labels:
        sampled_indices += random.sample(indices_by_label[lab], per_label)
    
    subset = train_data.select(sampled_indices)

    def perturb(ex):
        return {"text": augment_text(ex["text"])}

    perturbed_subset = subset.map(perturb)

    train_dataset = concatenate_datasets([train_data, perturbed_subset])

    return train_dataset, test_data, id2label, label2id, class_names



### Preprocess

In [5]:
COMPANY_LIST = [
    "google", "apple", "microsoft", "amazon", "facebook", "tesla",
    "oracle", "ibm", "intel", "nvidia", "qualcomm", "sap",
    "salesforce", "uber", "airbnb", "twitter", "meta", "snap",
    "zoom", "palantir"
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def mask_text(text: str) -> str:
    t = text.replace("\n", " ").strip().lower()
    t = ''.join('[NUM]' if ch.isdigit() else ch for ch in t)
    for comp in COMPANY_LIST:
        t = re.sub(rf"\b{comp}\b", '[COMPANY]', t)
    return t


### Train

In [6]:
accuracy_metric = load("accuracy")

def tokenize_function(examples, tokenizer):
    examples["text"] = [preprocess(text) for text in examples["text"]]
    examples["text"] = [mask_text(text) for text in examples["text"]]
    examples["text"] = [text.replace("\n", " ") for text in examples["text"]]

    tokenizer_resp = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
        return_tensors="pt",
    )
    examples["input_ids"] = tokenizer_resp["input_ids"]
    examples["attention_mask"] = tokenizer_resp["attention_mask"]
    return examples


def load_tokenizer(model_id):
    """
    Load the tokenizer for the specified model ID.
    
    Args:
        model_id (str): The model ID for loading the tokenizer.
    
    Returns:
        RobertaTokenizer: The loaded tokenizer.
    """
    return RobertaTokenizer.from_pretrained(model_id)

def load_model(model_id, num_labels, id2label, config):
    """
    Load the model for sequence classification.
    
    Args:
        model_id (str): The model ID for loading the model.
        num_labels (int): The number of labels for classification.
        id2label (dict): Mapping from label IDs to label names.
    
    Returns:
        RobertaForSequenceClassification: The loaded model.
    """

    model = RobertaForSequenceClassification.from_pretrained(
        model_id,
        num_labels=num_labels,
        id2label=id2label,
    ) 

    model = get_peft_model(model, config)

    return model

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

def train():
    device = "cuda:7" if torch.cuda.is_available() else "cpu"
    num_samples = 40000
    model_id = "roberta-base"
    num_labels = 4
    train_batch_size = 64
    test_batch_size = 32

    # Define the LoRA configuration
    lora_config = LoraConfig(
        r=4,
        lora_alpha=8,
        target_modules=["self_attn.value", "self_attn.query", "self_attn.key", "output.dense"],
        lora_dropout=0.1,
        bias='none',
        task_type=TaskType.SEQ_CLS,
        use_dora=True
    )
    
    # Load the augmented dataset
    train_dataset, test_dataset, id2label, label2id, class_names = get_augmented_agnews_data(num_samples)

    # Load the tokenizer and model
    tokenizer = load_tokenizer(model_id)
    model = load_model(model_id, num_labels, id2label, lora_config)
    model = model.to(device)

    # Tokenize the dataset
    train_dataset = train_dataset.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True,
        remove_columns=["text"],
    )

    test_dataset = test_dataset.map(
        lambda x: tokenize_function(x, tokenizer),
        batched=True,
        remove_columns=["text"],
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

    trainer_args = TrainingArguments(
        output_dir="../Models",
        overwrite_output_dir=True,
        max_steps=1000,
        logging_strategy="steps",
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        learning_rate=2e-4,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=test_batch_size,
        weight_decay=0.01,
        optim="adamw_torch",
        label_names=["label"],
        report_to="wandb",
        run_name="run_1"
    )

    trainer = Trainer(
        model=model,
        args=trainer_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.evaluate(eval_dataset=test_dataset)

    print(f"Training Done, Plotting losses and accuracies ...")

    os.makedirs("Models", exist_ok=True)
    trainer.save_model("Models")

    from sklearn.metrics import accuracy_score
    predictions = trainer.predict(test_dataset)
    y_pred = predictions.predictions[1]
    pred = y_pred.argmax(-1)
    accuracy = accuracy_score(test_dataset["label"], pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")

Using the latest cached version of the module from /mnt/export/arun/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--accuracy/f887c0aab52c2d38e1f8a215681126379eca617f96c447638f751434e8e65b14 (last modified on Mon Apr 21 21:25:25 2025) since it couldn't be found locally at evaluate-metric--accuracy, or remotely on the Hugging Face Hub.


In [7]:
train()

Using the latest cached version of the dataset since ag_news couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /mnt/export/arun/.cache/huggingface/datasets/ag_news/default/0.0.0/eb185aade064a813bc0b7f42de02595523103ca4 (last modified on Wed Apr 23 22:00:23 2025).


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: avp3799 (kai-xu-new-york-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Step,Training Loss,Validation Loss
50,0.942600,No log
100,0.351300,No log
150,0.309700,No log
200,0.301700,No log
250,0.290300,No log
300,0.275600,No log
350,0.272400,No log
400,0.265400,No log
450,0.257300,No log
500,0.259100,No log


/mnt/export/arun/envs/pirrs/lib/python3.11/site-packages/peft/utils/other.py:1110: UserWarning: Unable to fetch remote file due to the following error 401 Client Error: Unauthorized for url: https://huggingface.co/roberta-base/resolve/main/config.json (Request ID: Root=1-68096565-6f593042230e1b7f47112bf2;a9ed03fb-a597-4584-b69a-f2b7723d931d)

Invalid credentials in Authorization header - silently ignoring the lookup for the file config.json in roberta-base.
  warnings.warn(
/mnt/export/arun/envs/pirrs/lib/python3.11/site-packages/peft/utils/save_and_load.py:236: UserWarning: Could not find a config file in roberta-base - will assume that the vocabulary was not modified.
  warnings.warn(
/mnt/export/arun/envs/pirrs/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/mnt/export/arun/envs/pirrs/lib/python3.11/site-packages/peft/

Training Done, Plotting losses and accuracies ...


/mnt/export/arun/envs/pirrs/lib/python3.11/site-packages/peft/utils/other.py:1110: UserWarning: Unable to fetch remote file due to the following error 401 Client Error: Unauthorized for url: https://huggingface.co/roberta-base/resolve/main/config.json (Request ID: Root=1-6809698c-3af469ac3f622ee36b488e33;52deca51-b469-4c90-ab08-429a4aae3314)

Invalid credentials in Authorization header - silently ignoring the lookup for the file config.json in roberta-base.
  warnings.warn(
/mnt/export/arun/envs/pirrs/lib/python3.11/site-packages/peft/utils/save_and_load.py:236: UserWarning: Could not find a config file in roberta-base - will assume that the vocabulary was not modified.
  warnings.warn(


Accuracy: 92.42%
